# LH2 흡입배관 대리모델: 200개 파일럿
동일 조건 모델 비교, 데이터 누수 방지, 물리 오차 분리와 최종 테스트 평가. 실험 설계와 한계는 README.md에 정리했습니다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if (ROOT / '.local_deps').exists():
    sys.path.insert(0, str(ROOT / '.local_deps'))
import lh2_pipeline as study
import pandas as pd
DATA = ROOT / 'data/dataset_ai.csv'
OUT = ROOT / 'outputs/pilot'


## 1. 데이터 감사
결측·중복·로그 변환·단위·압력 구성식을 검사합니다. 입력 전용 1만 개 파일은 합치지 않습니다.

In [ ]:
df = study.load_data(DATA)
display(df.describe().T)

## 2. 개발 데이터에서 모델 선택
160개 개발 자료에서 동일 5-fold × 3회 CV를 사용합니다. 튜닝 모델당 6개 후보, seed 2026. 기존 실험은 재학습하지 않습니다.

In [ ]:
if not (OUT / 'manifest.json').exists():
    study.development(DATA, OUT, learning=True)
display(pd.read_csv(OUT / 'cv_summary.csv'))

## 3. 학습 곡선과 수렴 확인
테스트셋을 사용하지 않습니다. CV는 모델 선택용이며 독립적인 최종 성능 추정치가 아닙니다. 수렴 경고를 manifest에서 확인합니다.

In [ ]:
import json
from IPython.display import Image, display
manifest = json.loads((OUT / 'manifest.json').read_text(encoding='utf-8'))
print('Selected:', manifest['selected'])
print('Recorded warnings:', len(manifest['warnings']))
for target in study.TARGETS:
    for suffix in ['__learning.png', '__mlp_history.png']:
        image = OUT / (target + suffix)
        if image.exists():
            display(Image(filename=str(image)))

## 4. 최종 테스트: 모델 확정 후 한 번만
선정 모델과 사전 지정 MLP를 평가합니다. 결과를 보고 재튜닝하지 않습니다. 완료한 실험은 저장된 표만 확인합니다.

In [ ]:
if not (OUT / 'final_test_metrics.csv').exists():
    study.finalize(DATA, OUT)
display(pd.read_csv(OUT / 'final_test_metrics.csv'))
print((OUT / 'final_pressure_metrics.json').read_text(encoding='utf-8'))

## 5. 해석과 한계
최종 테스트는 40개뿐입니다. 수력식 근사오차와 AI 오차를 분리해서 보고합니다. 실제 설계 확정에는 솔버 재검증이 필요합니다. 열침입 물리 기준선, 참고논문 재현 ANN과 최적화는 추가 자료 확보 후 진행합니다.